In [ ]:
import pandas as pd
import numpy as np
from owlready2 import *
import re

from skmultilearn.model_selection import iterative_train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import joblib
import pickle

# Import processed publications

In [ ]:
# Code to import the processed publications from a CSV file.
# Checkpoint data
# The list of strings in the 'PaNET_labels_set' and 'PaNET_IRIs_set' columns will be read as strings, so we need to convert them back to lists.

import ast

filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/checkpoint_data/df_filtered.csv'
df_filtered = pd.read_csv(filepath,encoding='utf-8')

df_filtered['PaNET_labels_set'] = df_filtered['PaNET_labels_set'].apply(ast.literal_eval)   # Convert the string representation of a list back into an actual list
df_filtered['PaNET_IRIs_set'] = df_filtered['PaNET_IRIs_set'].apply(ast.literal_eval)   # Convert the string representation of a list back into an actual list

# Load PaNET ontology

In [ ]:
# Load the ontology
ontology_path='/Users/fdp54928/Documents/PaNET-classifier/Data/owlapi.xrdf'
onto=get_ontology(ontology_path).load()

# Run reasoner
with onto:
    sync_reasoner()  # Runs reasoning and updates inferred relationships

#  Explore the ontology to find all techniques (descendants of PaNET00001)
onto_ls = list(onto.classes())
panet00001 = onto.search(iri='http://purl.org/pan-science/PaNET/PaNET00001')[0]
technique_terms = [cls for cls in panet00001.descendants() if cls.iri!='https://www.wikidata.org/wiki/Q133900']
technique_label = [cls.label[0].strip().lower() for cls in technique_terms]
technique_iri = [cls.iri for cls in technique_terms]
technique_altLabel = [cls.altLabel for cls in technique_terms]

print('Number of technique classes:',len(technique_iri))

# Training/Testing/Validation split

In [ ]:
# Stack the titles and abstracts as two columns: Column 0 = Title, Column 1 = Abstract, Row i = ith Publication
# This is our input, X
dois = df_filtered['DOI']
titles = df_filtered['Title of Paper']
abstracts = df_filtered['Abstract']
X = np.column_stack((dois,titles,abstracts))

In [ ]:
# Binarize the labels, Y
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df_filtered['PaNET_IRIs_set'].to_list())

print('Number of labels:',len(mlb.classes_))
print('Number of publications:',len(Y))

In [ ]:
# Split (iterative_train_test_split will keep the row relationship intact)
# We want to do a two-step split to create Train, Validation, and Test sets in a roughly 80-10-10 split while preserving the label distribution across all three sets, 
# which is crucial for multi-label classification tasks.

# First split: Train vs Test
X_train, Y_train, X_test, Y_test = iterative_train_test_split(X, Y, test_size=0.1)

# Second split: Train vs Validation (from the training set)
X_train, Y_train, X_val, Y_val = iterative_train_test_split(X_train, Y_train, test_size=0.11)  # 0.11 x 0.9 = 0.099 of the total data for validation, 0.801 for training

In [ ]:
# Save the Binarizer
# This includes .classes_ and internal mapping
filepath = '/Users/fdp54928/Documents/PaNET-classifier/baseline_model/data/'
joblib.dump(mlb, filepath + 'binarizer.pkl')

# Combine X and Y into DataFrames for storage
# This ensures that row i of X is always row i of Y
train_df = pd.concat([
    pd.DataFrame(X_train, columns = ['DOI','Title','Abstract']).reset_index(drop=True), 
    pd.DataFrame(Y_train, columns=mlb.classes_)
], axis=1)

test_df = pd.concat([
    pd.DataFrame(X_test,columns = ['DOI','Title','Abstract']).reset_index(drop=True), 
    pd.DataFrame(Y_test, columns=mlb.classes_)
], axis=1)

val_df = pd.concat([
    pd.DataFrame(X_val,columns = ['DOI','Title','Abstract']).reset_index(drop=True), 
    pd.DataFrame(Y_val, columns=mlb.classes_)
], axis=1)

# Save Train/Test/Validation splits to Parquet
train_df.to_parquet(filepath + 'train_set.parquet', index=False)
test_df.to_parquet(filepath + 'test_set.parquet', index=False)
val_df.to_parquet(filepath + 'val_set.parquet', index=False)

# Build ancestor map
We need to build an ancestor map to calculate hF1 score during training/inference.

In [ ]:
# Build ancestor map from PaNET
def build_ancestor_map(onto, panet_iris):
    excluded_iris = {
    'http://purl.org/pan-science/PaNET/PaNET00001', 
    'http://www.w3.org/2002/07/owl#Thing',
    'https://www.wikidata.org/wiki/Q133900'
    }
    ancestor_map = {}
    for iri in panet_iris:
        cls = onto.search_one(iri=iri)
        if cls is not None:
            # Get all ancestors excluding owl:Thing
            ancestors = {a.iri for a in cls.ancestors() if a.iri not in excluded_iris and a.iri in panet_iris}
        else:
            ancestors = set()
        ancestor_map[iri] = ancestors
    return ancestor_map

In [ ]:
ancestor_map = build_ancestor_map(onto, mlb.classes_)

In [ ]:
# Save the ancestor map using pickle
filepath = '/Users/fdp54928/Documents/PaNET-classifier/baseline_model/data/'
with open(filepath + 'ancestor_map.pkl', 'wb') as f:
    pickle.dump(ancestor_map, f)

In [ ]:
# Pre-compute index-based ancestor sets for speed
ancestor_indices = []
for i, label in enumerate(mlb.classes_):
    indices = {j for j, a in enumerate(mlb.classes_) if a in ancestor_map[label]}
    ancestor_indices.append(frozenset(indices))

In [ ]:
# Save the ancestor indices using pickle
filepath = '/Users/fdp54928/Documents/PaNET-classifier/data/baseline_model/'
with open(filepath + 'ancestor_indices.pkl', 'wb') as f:
    pickle.dump(ancestor_indices, f)